# Simulation -- test every scan mode on a known topography

Runs **exactly the same scan modes as `Run_SelfDriving_Scan.ipynb`** -- same planners,
executor, live view, summary and saved file (all from `selfdriving_afm/modes.py`) --
against a sample whose true topography is known, so you can see where a mode works,
where it fails, what it would cost on the instrument, and try changes before touching
the microscope. The only difference from a real run is the backend (`SimulateBackend`).

Run top to bottom:
1. **Sample** -- describe the topography in text (e.g. "2 triangles and 1 square of height
   0.5 and size 1 V") **or** load a channel of an `.ibw` image.
2. **Configuration** -- by default the Run notebook's own configuration cell, so you test
   the settings you'd actually use; then any overrides for this simulation.
3. **Run one mode** with the live view, then score it against the ground truth.
4. **Look inside** the planner (tiles, features, the scan log) to see *why*.
5. **Compare modes / settings** side by side on the same sample (no live view, faster).
6. **Sweep** one parameter.
7. **Save** -- same `.npz` format as a real run (plus the ground truth), readable by
   `Analyze_SelfDriving_Scan.ipynb`.

**What the simulation does not reproduce** (printed again in section 3 when it applies):
PyScanner's own scan types (`grid_refine.scan_generator = "pyscanner_defined"`) are
simulated with `pyscanner_scans.predict_path`, which the 2026-09-24 hardware probe showed
differs from PyScanner (raster trace+retrace, Lissajous frequencies, spiral return,
rotation direction); `readback.mode = "image"` is simulated as if it were `"raw"`; and
the per-`do_scan` overhead only appears in the *estimated* instrument time.

In [ ]:
import warnings
warnings.filterwarnings("ignore")  # sklearn GP kernel-bound convergence warnings are benign

import pathlib
import numpy as np
import matplotlib
%matplotlib inline
import matplotlib.pyplot as plt

from selfdriving_afm import modes
from selfdriving_afm import simulation as sim
from selfdriving_afm.config import ExperimentConfig
from selfdriving_afm.backends.simulate_backend import SimulateBackend
from selfdriving_afm.topography import sample_from_text, sample_from_ibw

print("Scan modes available (same list as the Run notebook):")
print(modes.list_modes())

## 1. The sample

**`SAMPLE_SOURCE = "text"`** -- one feature group per line: *count, shape, settings*.

| | |
|---|---|
| shapes | `square`, `rectangle`, `triangle`, `circle` (disk/cylinder), `ellipse`, `ring`, `bump` (Gaussian), `pyramid`, `step` (terrace edge), `polygon` |
| `height=` | in Z units (like AI0); negative = a pit. Default 0.3 |
| `size=` | lateral size in V: one number (square side, circle diameter, equilateral triangle side, bump FWHM) or `WxH` (rectangle, ellipse, isosceles triangle base x height) |
| `at=(x, y)` | centre in V; leave it out and the feature is placed at random (reproducibly, set by `seed`), not touching the others |
| `angle=` | rotation in degrees (step: the direction the ground rises) |
| `edge=` | sidewall width in V (0 = vertical walls) -- like a blunt tip |
| `width=` | ring wall width; `points=(x1,y1),(x2,y2),...` polygon vertices |
| global lines | `field 4` (half-width of the sample square, V), `base -5.7` (substrate level), `tilt 0.01 -0.02` (dz/dx dz/dy), `roughness 0.007 1 4` (smooth substrate texture: RMS, then shortest/longest wavelength in V), `seed 3`, `gap 0.3` (clearance between random features) |

Features add up (a square on a `step` is a stacked terrace). `#` starts a comment.

**`SAMPLE_SOURCE = "ibw"`** -- one channel of an Asylum `.ibw` image (e.g. the rasters in
`scan_data/`), laid over a `2 x IBW_HALF_SIZE_V` square in bias volts (the package works in
open-loop volts, so how many V the image spans is your choice), flattened, and rescaled
so its height range is `IBW_Z_RANGE` -- then thresholds tuned on text samples carry over.
Set `IBW_Z_SCALE` to a number instead (e.g. `1e9`: metres -> nm) to keep real heights.

The sample should cover the area your modes scan (`grid_refine.half_size_V`,
`edge_follow.explore_extent_V`, `edge_explorer.explore_extent_V`, ...): outside it, a text
sample is flat substrate and an image repeats its edge pixels.

**Default sample: the instrument run of 2026-10-05** (a hexagon and a second object, ~0.3 V
tall, on a sample tilted ~0.26 V over the 8 V area, with ~7 mV of substrate texture and
0.4 mV of noise -- all measured from that run's `.npz`). That tilt is what made the edge
explorer find 9 "features" for 2 real ones before the on-the-fly tilt correction
(`substrate.SubstrateModel`, README "Tilt"); section 3b checks the correction, section 5
compares with it switched off. The earlier mixed sample is kept, commented out.

In [ ]:
SAMPLE_SOURCE = "text"   # "text" or "ibw"

SAMPLE_TEXT = """
field 4                                   # the sample is 8 x 8 V, centred on (0, 0)
base -6.0                                 # substrate level (AI0 on the instrument: ~ -6.0 V)
tilt 0.005 0.033                          # sample tilt, dz/dx dz/dy -- measured on the 2026-10-05 run
roughness 0.007 1 4                       # substrate texture: 7 mV RMS, wavelengths 1-4 V (same run)
1 polygon    height=0.28 at=(1.7, 0.5) edge=0.03 points=(0.966,0.259),(0.259,0.966),(-0.707,0.707),(-0.966,-0.259),(-0.259,-0.966),(0.707,-0.707)   # the hexagon
1 rectangle  height=0.31 size=1.4x0.5 at=(0.8, -3.75) edge=0.03     # the object at the bottom
1 circle     height=0.30 size=1.2 at=(-2.3, 2.2) edge=0.03           # third shape (gp_path tuning, 2026-11-06)
"""
# the 2-feature sample (hexagon + rectangle only): delete the circle line above.
# the 5-feature tuning sample: add
#   1 triangle height=0.25 size=1.3 at=(-2.4, -1.6) edge=0.03
#   1 square   height=0.35 size=0.8 at=(2.6, 2.9) edge=0.03

# the earlier test sample (flat substrate, six features) -- swap it in to test the other modes:
# SAMPLE_TEXT = """
# field 4
# base 0
# tilt 0 0
# seed 4
# 2 triangles  height=0.5 size=1.2
# 1 square     height=0.5 size=1.0
# 1 rectangle  height=0.3 size=1.5x0.6 angle=30
# 2 circles    height=0.6 size=0.8
# """

# --- .ibw (SAMPLE_SOURCE = "ibw") ---
IBW_FILE = "scan_data/raster_scan_image0000.ibw"
IBW_CHANNEL = "HeightRetrace"      # the channels the file has are printed below
IBW_HALF_SIZE_V = 4.0              # the image spans +/- this many V (centred on IBW_CENTER_V)
IBW_CENTER_V = (0.0, 0.0)
IBW_FLATTEN = "plane+line"         # "none", "plane", "line" or "plane+line"
IBW_Z_SCALE = "auto"               # "auto" = rescale to IBW_Z_RANGE, or a number (e.g. 1e9: m -> nm)
IBW_Z_RANGE = 0.6                  # "auto": the 1-99% height range becomes this
IBW_Z_OFFSET = 0.0                 # added last (e.g. -5.7 for an AI0-like baseline)

# --- measurement noise (std of Z per sample, same units as height) ---
NOISE_STD = 0.0004   # measured on the instrument (2026-10-05, raw readback); the old default was 0.005
NOISE_SEED = 1

if SAMPLE_SOURCE == "text":
    sample = sample_from_text(SAMPLE_TEXT)
elif SAMPLE_SOURCE == "ibw":
    sample = sample_from_ibw(IBW_FILE, IBW_CHANNEL, half_size_V=IBW_HALF_SIZE_V, center_V=IBW_CENTER_V,
                             flatten_method=IBW_FLATTEN, z_scale=IBW_Z_SCALE, z_range=IBW_Z_RANGE,
                             z_offset=IBW_Z_OFFSET)
else:
    raise ValueError(f"SAMPLE_SOURCE must be 'text' or 'ibw', not {SAMPLE_SOURCE!r}")

print(sample.description)
fig, ax = plt.subplots(figsize=(6.5, 5.2), layout="constrained")
sample.plot(ax)
plt.show()

## 2. Configuration

By default this **runs the configuration cell of `Run_SelfDriving_Scan.ipynb`** (only that
cell -- nothing connects to hardware), so the simulation uses exactly the mode and
settings you'd run on the instrument: edit them there, re-run this cell. Set
`USE_RUN_NOTEBOOK_CONFIG = False` to start from the package defaults instead.

Then change anything for this simulation only in the overrides below -- any config field
works (`cfg.<section>.<field>`; see `selfdriving_afm/config.py`). A simulation is only as
fast as the number of samples: `tip_velocity_V_s` and `sample_rate_hz` set that, exactly
as on the instrument.

In [ ]:
USE_RUN_NOTEBOOK_CONFIG = True
RUN_NOTEBOOK = "Run_SelfDriving_Scan.ipynb"

if USE_RUN_NOTEBOOK_CONFIG:
    cfg, MODALITY, LIVE_VIEW_EXTENT_V = modes.load_run_config(RUN_NOTEBOOK)
    print(f"Configuration from {RUN_NOTEBOOK}: MODALITY = {MODALITY!r}")
else:
    cfg, MODALITY, LIVE_VIEW_EXTENT_V = ExperimentConfig(), "grid_refine", 0.45
    print("Package default configuration")

# --- overrides for this simulation only (uncomment / add as needed) ---
# MODALITY = "edge_follow"            # "edge_follow", "gp_path", "edge_explorer" or "grid_refine"
# cfg.velocity.tip_velocity_V_s = 50
# cfg.grid_refine.scan_generator = "user_defined"
# cfg.edge_follow.edge_detect.manual_threshold = 0.2
cfg.readback.mode = "raw"             # the simulator always returns exact per-sample Z

print(f"tip {cfg.velocity.tip_velocity_V_s:g} V/s at {cfg.velocity.sample_rate_hz:g} samples/s "
      f"-> one sample every {cfg.velocity.sample_spacing_V:.3g} V")
print(f"sample area: X {sample.bounds[0]:g}..{sample.bounds[1]:g}, Y {sample.bounds[2]:g}..{sample.bounds[3]:g} V")

## 2b. Priors -- what you expect on this sample

The same step as in the Run notebook: asked **before the scan starts** (press Enter to keep the value in brackets):
- the **smallest and largest expected feature**, in V -- a feature's narrowest width (a
  disk's diameter, a line's width);
- the **most the scan may explore**: a square (half-width and center, in V).

They set every mode's area and size parameters (`selfdriving_afm/priors.py`) -- these
are no longer set in the configuration cell:
- **edge_follow**: the first search spiral covers the whole area, its turns half the
  smallest feature apart (no feature fits between two turns); later spirals fill what's
  left, inside the area. Tracing circles are sized from the curvature each circle
  measures (the angle its on-feature arc spans), between 0.25 x the smallest and
  0.75 x the largest feature -- small round features get small circles, long straight
  edges big ones (each circle costs ~3 s of do_scan overhead).
- **grid_refine**: the sparse scan covers the area, lines 0.7 x the smallest feature apart.
- **edge_explorer**: explores the area from its center, sweeps ~the smallest feature apart;
  follows each edge it crosses until it closes, then scans the inside with lines 0.7 x the
  smallest feature apart (holes found there are followed too).
- **live view**: always shows the whole area to explore, in every frame.
- **gp_path**: stays inside the area; GP length scale from the feature sizes.

Here the defaults are the sample's true priors (`priors.from_sample`); `USE_PRIORS = False` runs without them, for comparison. The printout says what was set, with the first spiral's size and time.


In [ ]:
from selfdriving_afm import priors

# the defaults offered when asked: this sample's TRUE priors (its smallest / largest
# feature and its field) -- change them to see what a wrong prior does, e.g.
# PRIOR_DEFAULTS = priors.ScanPriors(feature_min_V=0.5, feature_max_V=2.0, area_half_width_V=4.0, area_center_V=(0, 0))
PRIOR_DEFAULTS = priors.from_sample(sample)
ASK_PRIORS = True       # False = use PRIOR_DEFAULTS without asking
USE_PRIORS = True       # False = no priors: the configuration's own settings (the Run notebook's
                        #   configuration cell no longer sets the areas -> package defaults, small)

PRIORS = None
if USE_PRIORS:
    PRIORS = priors.ask(PRIOR_DEFAULTS, interactive=ASK_PRIORS)
    print("Priors:", PRIORS)
    for note in priors.apply(cfg, PRIORS):   # sets every mode's area + size parameters in cfg
        print(" ", note)
    LIVE_VIEW_EXTENT_V = PRIORS.area_half_width_V
else:
    cfg.live_view.fixed_bounds_V = tuple(sample.bounds)
# the live view always shows this whole area, in every frame (the priors' area, or the
# sample's field without priors); set cfg.live_view.fixed_bounds_V = None to let it grow
# with the scanned area instead


## 3. Run one mode, with the live view

`modes.build` + `modes.run` -- the calls the Run notebook makes, on a `SimulateBackend`
reading this sample. The live view (same as on the instrument, with the ground truth
under the path) updates in this cell's output; the summary is printed at the end.

**Every step is shown** (`LIVE_EVERY_STEP = True`): all panels are redrawn after every
waveform -- each spiral, circle, segment, tile batch, and each move between them -- with
the figure title saying which step it was and what it was for. (On the instrument the
live view instead throttles its redraws so it never slows the acquisition; with modes
that send few, long waveforms that can mean nothing new appears until the end.)
Drawing a frame takes ~1 s (~2 s with the raster row: the figure itself ~0.8 s, plus a CS
reconstruction per row), so a run with 150 steps takes ~5 min instead of ~15 s.
`SKIP_MOVES = True` doesn't draw a frame after the moves between scans (their path still
shows in the next frame) -- about half the steps of the spiral/circle modes -- or set
`LIVE_EVERY_STEP = False` for the instrument's throttled live view.

**Step-by-step replay** (`REPLAY_STEPS = True`): once the run is finished, the same cell
prints the whole figure as it was after each step, in order, so you can trace afterwards
what the mode did and when. Runs with hundreds of steps make a long output (~300 kB per
frame; the total is printed first) -- `REPLAY_EVERY = 5` shows every 5th step, and
`SAVE_FRAMES_TO` writes every frame as a PNG file instead (or as well), to flip through
in an image viewer.

**Animation** (`SAVE_GIF = True`): all the frames saved as one looping GIF, e.g. for
slides or to send around -- `GIF_FPS` sets the speed, `GIF_EVERY = 3` keeps every 3rd
frame (shorter, smaller file), `GIF_MAX_WIDTH` scales frames down. Size: ~0.2 MB per
frame at 1200 px, ~0.12 MB at 800 px -- a 150-step run is ~30 MB, so `GIF_EVERY = 3`
is a good start for long runs. Saved to `scan_data/simulation/` unless `GIF_PATH` says
otherwise.

**Second row: the classic raster with the same budget** (`RASTER_COMPARE = True`) --
at every redraw, a plain raster scan of the same area (every line left to right, one
after another from the bottom, no self-driving) with the same number of measured
pixels as the mode so far, on the same pixel grid: its lines, its binned AI0 and its
reconstruction (same color scale as the first row). So you see, at each moment,
whether the smart path is actually better than just rastering.
- `RASTER_RECON = "raster_columns"`: the raster's gaps are filled column by column
  between its lines (PCHIP -- the best method for raster data; CS aliases on regularly
  skipped lines). `None` = the same method as the mode (`cfg.live_view.reconstruction`).
- `RASTER_LAYOUT = "sparse"`: the raster's lines spread evenly over the whole area (a
  coarser raster) -- the fair comparison for the reconstruction. `"progressive"`: a
  full-resolution raster (one line per pixel row) stopped part-way at the same budget.
- `RASTER_MATCH = "pixels"`: same number of measured pixels. `"path"`: same tip path
  length instead (trace + retrace), i.e. the same scan time at the same tip speed.

The next cell **scores the run against the ground truth** over the sample area:
- **bad pixels** -- reconstruction pixels off by more than `BAD_FRAC` x the sample's
  height range (all pixels, and near edges only),
- **edges visited** -- fraction of the true edges a sample passed near (within 2 pixels);
  per feature too, and features with less than half visited are listed as **MISSED**,
- **cost** -- samples, waveforms (= `do_scan` calls), waveform time, and the **estimated
  instrument time** (waveforms + ~3 s per `do_scan` + readback, as measured on the instrument).

With the raster comparison on, the classic raster with the same budget is scored the
same way (second row of panels, and a "self-driving is BETTER / WORSE" line).

In [ ]:
RASTER_COMPARE = True      # second row: the classic raster with the same budget
RASTER_LAYOUT = "sparse"   # "sparse" (lines spread over the whole area) or "progressive" (full resolution, stopped part-way)
RASTER_MATCH = "pixels"    # "pixels" (same measured pixels) or "path" (same tip path = same scan time)
RASTER_RECON = "raster_columns"  # raster's reconstruction: column-wise PCHIP (best for rasters); None = mode's method

LIVE_EVERY_STEP = True     # redraw ALL panels after EVERY waveform (the instrument throttles redraws instead)
SKIP_MOVES = False         # True: no frame after the moves between scans (their path still shows in the
                           #   next frame) -- ~2x faster for spiral/circle modes, where half the steps are moves
REPLAY_STEPS = True        # after the run: show the figure as it was after each step, in order
REPLAY_EVERY = 1           # replay every N-th step (the final frame is always shown)
SAVE_FRAMES_TO = None      # e.g. "scan_data/simulation/frames" -> also write every frame as a PNG
SAVE_GIF = True           # True: also save all the frames as one animated GIF (loops)
GIF_PATH = None           # None = scan_data/simulation/<mode>_<date_time>.gif
GIF_FPS = 4               # frames per second (the last frame is held 3 s)
GIF_EVERY = 1             # use every N-th frame (bigger = shorter, smaller file; ~0.2 MB per frame)
GIF_MAX_WIDTH = 1200      # frames scaled down to at most this many pixels wide

for note in sim.model_notes(cfg, MODALITY):
    print("NOTE:", note, "\n")

raster = (sim.EquivalentRaster(sample, cfg, NOISE_STD, layout=RASTER_LAYOUT, match=RASTER_MATCH,
                               seed=NOISE_SEED, reconstruction=RASTER_RECON)
          if RASTER_COMPARE else None)
backend = SimulateBackend(sample, noise_std=NOISE_STD, seed=NOISE_SEED)
session = modes.build(MODALITY, cfg, backend, live=True, live_extent=LIVE_VIEW_EXTENT_V,
                      ground_truth_fn=sample, compare=raster,
                      priors=PRIORS)
executor, planner, live = session.executor, session.planner, session.live
live.every_step = LIVE_EVERY_STEP
live.skip_moves = SKIP_MOVES
live.keep_frames = REPLAY_STEPS or SAVE_FRAMES_TO is not None or SAVE_GIF
result = modes.run(session)

if SAVE_FRAMES_TO:
    live.save_frames(SAVE_FRAMES_TO)
if SAVE_GIF:
    import time
    gif = GIF_PATH or pathlib.Path("scan_data/simulation") / f"{MODALITY}_{time.strftime('%Y%m%d_%H%M%S')}.gif"
    live.save_gif(gif, fps=GIF_FPS, every=GIF_EVERY, skip_connectors=SKIP_MOVES, max_width=GIF_MAX_WIDTH)
if REPLAY_STEPS:
    live.show_frames(every=REPLAY_EVERY, skip_connectors=SKIP_MOVES)

In [ ]:
BAD_FRAC = 0.2          # a pixel is "bad" if it's off by more than this x the sample's height range
EVAL_BOUNDS = None      # None = the sample area, or (xmin, xmax, ymin, ymax) in V
EVAL_PIXELS = 200

ev = sim.evaluate(session, sample, eval_bounds=EVAL_BOUNDS, n=EVAL_PIXELS, bad_frac=BAD_FRAC, raster=raster)
print(sim.format_evaluation(ev))
sim.plot_evaluation(ev)
plt.show()

## 3b. Tilt correction: what the planner learned about the substrate

`edge_follow` / `edge_explorer` decide on/off-feature from the height above a substrate
plane they fit while scanning (`planner.substrate`, `substrate.SubstrateModel`; README
"Tilt"). This compares the plane they ended with to the sample's true tilt, shows how the
on/off threshold evolved (the jump = the moment the features' height was known and the
threshold moved into the gap between substrate and features), and the histogram of
heights above the plane it split. On the instrument the same numbers are in
`print(planner.substrate.summary())` and saved in the `.npz` (`substrate_json`,
`substrate_threshold_history`).

In [ ]:
from selfdriving_afm.topography import parse_text

model = getattr(planner, "substrate", None)
if model is None or not model.fitted:
    print(f"{MODALITY} has no substrate model (only edge_follow and edge_explorer use one)")
else:
    mad_k = getattr(cfg, MODALITY).edge_detect.mad_k
    true = parse_text(SAMPLE_TEXT)[1] if SAMPLE_SOURCE == "text" else {}
    tx, ty = true.get("tilt", (0.0, 0.0))
    c = model.coef
    print(f"fitted substrate: level {c[0]:.4f} at (0, 0), slope {c[1]:.4f}, {c[2]:.4f} (dz/dx, dz/dy)"
          if model.degree >= 1 else f"fitted substrate: constant level {c[0]:.4f} (tilt_correction=False)")
    if SAMPLE_SOURCE == "text":
        print(f"true substrate:   level {true.get('base', 0.0):.4f} at (0, 0), slope {tx:.4f}, {ty:.4f}")
    print(f"roughness (tilt-corrected substrate) {model.sigma:.4f}, final threshold {model.threshold:.4f} "
          f"({'in the gap between substrate and features' if model.gap_active else f'{mad_k:g} x roughness'})")

    hist = np.array([[h[0], h[1], h[2], h[3]] for h in model.history])
    pts = executor.all_points()
    hgt = model.height(pts[::50, 0], pts[::50, 1], pts[::50, 2])
    fig, ax = plt.subplots(1, 2, figsize=(13, 3.8), layout="constrained")
    ax[0].plot(hist[:, 0], hist[:, 1], label="threshold")
    ax[0].plot(hist[:, 0], cfg.edge_explorer.edge_detect.mad_k * hist[:, 2], "--", label="mad_k x roughness")
    ax[0].set_xlabel("samples seen by the model (thinned)"); ax[0].set_ylabel("height above substrate")
    ax[0].set_title("On/off-feature threshold during the run"); ax[0].legend()
    ax[1].hist(hgt, bins=200, log=True, color="tab:gray")
    ax[1].axvline(model.threshold, color="tab:red", label=f"threshold {model.threshold:.3f}")
    ax[1].axvline(-model.threshold, color="tab:red")
    ax[1].set_xlabel("height above the fitted substrate"); ax[1].set_ylabel("samples (log)")
    ax[1].set_title("Every measured sample: substrate peak at 0, features to the right"); ax[1].legend()
    plt.show()


## 4. Look inside the planner

Why did it do that? The scan-by-scan log (what every waveform was for), and the
planner's own state: grid+refine's detection maps and tiles per level, the features the
edge planners found. Everything is on `planner` / `executor.history` to dig further.

In [ ]:
N_LOG_LINES = 40
log = live.log if live is not None else []
print(f"{len(log)} waveforms sent; first {min(N_LOG_LINES, len(log))}:")
print("\n".join(log[:N_LOG_LINES]))

if MODALITY == "grid_refine":
    from matplotlib.patches import Rectangle
    b = planner.bounds
    maps = [(planner.binned, "Binned (measured pixels)", "copper"),
            (planner.roughness, "Local variation (tilt removed)", "viridis"),
            (planner.flagged.astype(float) if planner.flagged is not None else None,
             "Flagged at the last detection", "gray")]
    fig, axes = plt.subplots(1, 4, figsize=(19, 4.4), layout="constrained")
    for ax, (img, title, cm) in zip(axes, maps):
        if img is not None:
            im = ax.imshow(img, origin="lower", extent=b, cmap=cm, interpolation="nearest")
            fig.colorbar(im, ax=ax, shrink=0.85)
        ax.set_title(title)
    axes[3].imshow(sample.image(300, b), origin="lower", extent=b, cmap="copper", alpha=0.7)
    colors = plt.cm.autumn(np.linspace(0, 1, max(len(planner.levels), 2)))
    for lv, c in zip(planner.levels[1:], colors[1:]):
        for i, r in enumerate(lv.rects):
            axes[3].add_patch(Rectangle((r.x0, r.y0), r.x1 - r.x0, r.y1 - r.y0, fill=False, ec=c, lw=0.8,
                                        label=f"level {lv.index}: {len(lv.rects)} tiles" if i == 0 else None))
    if len(planner.levels) > 1:
        axes[3].legend(fontsize=8, loc="upper right")
    axes[3].set_title("Refinement tiles over the ground truth")
    for ax in axes:
        ax.set_aspect("equal"); ax.set_xlabel("X (V)")
    plt.show()
    for lv in planner.levels:
        thr = "" if lv.threshold is None else f", threshold {lv.threshold:.3g} (noise {lv.noise:.3g})"
        print(f"level {lv.index}: spacing {lv.spacing_V:.3g} V, {len(lv.rects)} tile(s), "
              f"{100 * lv.area_fraction:.1f}% of the area{thr}")
elif MODALITY in ("edge_follow", "edge_explorer"):
    for f in result:
        n = len(f.circle_steps) if MODALITY == "edge_follow" else len(f.points)
        print(f"feature {f.id}: {n} {'circles' if MODALITY == 'edge_follow' else 'edge crossings'}, "
              f"closed_loop={f.closed_loop}")
    if MODALITY == "edge_explorer":
        print(f"segments: {len(planner.segments)} "
              f"({sum(s.mode == 'follow' for s in planner.segments)} following an edge)")
elif MODALITY == "gp_path":
    print(f"{len(result)} segments; GP kernel after the last fit: {planner.gp.kernel_}")

## 5. Compare modes / settings on the same sample

Every variant runs on this sample with the configuration above (no live view -- faster),
and gets one table row and one row of panels (ground truth + path | measured | recon |
error). A variant is either a mode name, or `("label", "mode", {"section.field": value, ...})`
to try settings -- e.g. the same mode with a different detection threshold. A variant
that crashes is reported in the table instead of stopping the comparison.

`gp_path` re-fits an exact GP on everything measured -- with the Run notebook's settings
it's the slowest by far (minutes); drop it from the list for quick iterations.

The two `NO tilt correction` variants switch off the substrate plane and the calibration
circle (`edge_detect.tilt_correction = False`: one constant substrate level) -- on the
tilted default sample they show what the correction buys.

In [ ]:
VARIANTS = [
    "edge_follow",
    ("edge_follow, NO tilt correction", "edge_follow", {"edge_follow.edge_detect.tilt_correction": False,
                                                         "edge_follow.calibration_radius_V": None}),
    "edge_explorer",
    ("edge_explorer, NO tilt correction", "edge_explorer", {"edge_explorer.edge_detect.tilt_correction": False,
                                                             "edge_explorer.calibration_radius_V": None}),
    "grid_refine",
    ("grid_refine, own raster paths", "grid_refine", {"grid_refine.scan_generator": "user_defined",
                                                       "grid_refine.sparse_pattern": "raster",
                                                       "grid_refine.refine_pattern": "raster"}),
    "gp_path",
]

# raster=...: the table also scores the classic raster with each variant's budget (None = off)
evs = sim.compare(VARIANTS, cfg, sample, noise_std=NOISE_STD, seed=NOISE_SEED, bad_frac=BAD_FRAC,
                  eval_bounds=EVAL_BOUNDS, n=EVAL_PIXELS,
                  raster=dict(layout=RASTER_LAYOUT, match=RASTER_MATCH, reconstruction=RASTER_RECON) if RASTER_COMPARE else None)
for e in evs:
    if "error" in e:
        print(f"\n{e['label']} FAILED:\n{e['trace']}")

## 5b. Compare all GP presets on this sample

Every `gp_path` preset (selfdriving_afm/gp_presets.py) on the sample above, with the same
path budget (`cfg.gp_path.max_path_length_V` from the Run notebook), plus the classic raster
with the same budget. Each runs without the live view: about 0.5-3 min per preset
(longer budgets take longer). Pick a subset in `GP_COMPARE` for a quick look.


In [ ]:
from selfdriving_afm.gp_presets import GP_PRESETS, variants as gp_variants

GP_COMPARE = list(GP_PRESETS)       # or e.g. ["pure_gp", "pure_gp_smooth", "original"]
GP_BUDGET_V = cfg.gp_path.max_path_length_V   # same tip travel for every preset (None = no limit)

evs_gp = sim.compare(gp_variants(GP_COMPARE), cfg, sample, noise_std=NOISE_STD, seed=NOISE_SEED, bad_frac=BAD_FRAC,
                     eval_bounds=EVAL_BOUNDS, n=EVAL_PIXELS,
                     raster=dict(layout=RASTER_LAYOUT, match=RASTER_MATCH, reconstruction=RASTER_RECON) if RASTER_COMPARE else None)
for e in evs_gp:
    if "error" in e:
        print(f"\n{e['label']} FAILED:\n{e['trace']}")


## 6. Sweep one parameter

One mode, one config field over several values -- plots the chosen metrics against it
(e.g. detection threshold vs. quality vs. estimated instrument time). Off by default.

In [ ]:
RUN_SWEEP = False
SWEEP_MODE = "grid_refine"
SWEEP_PARAM = "grid_refine.detect_k"      # any config field, as "section.field"
SWEEP_VALUES = [3.0, 5.0, 8.0, 12.0]
SWEEP_METRICS = ("bad_px", "edge_cov", "est_instrument_s")

if RUN_SWEEP:
    sweep_evs = sim.sweep(SWEEP_MODE, SWEEP_PARAM, SWEEP_VALUES, cfg, sample, metrics=SWEEP_METRICS,
                          noise_std=NOISE_STD, seed=NOISE_SEED, bad_frac=BAD_FRAC, eval_bounds=EVAL_BOUNDS,
                          n=EVAL_PIXELS)

## 7. Save the section-3 run (optional)

Set `SAVE_RUN = True` to keep it -- files are big (every sample, ~50 bytes each).
`modes.save` -- the same `.npz` + `.png` a real run writes -- plus the ground truth
(`gt_Z` over `gt_bounds`), the sample description and the scores. Saved in
`scan_data/simulation/`, so `Analyze_SelfDriving_Scan.ipynb` doesn't pick it up as the
newest real run; to replay one there, set its `SCAN_FILE` to the saved path.

In [ ]:
SAVE_RUN = False    # off by default: a run is saved at full resolution -- ~50 bytes per sample,
                    # e.g. ~2 GB for a 40 M-sample grid_refine run (and it syncs to Dropbox)
SIM_OUTPUT_DIR = pathlib.Path("scan_data") / "simulation"

print(f"This run: {len(executor.all_points()):,} samples -> ~{50 * len(executor.all_points()) / 1e9:.1f} GB .npz")
if SAVE_RUN:
    extra = dict(simulation=True, sample_source=SAMPLE_SOURCE, sample_description=sample.description,
                 sample_text=SAMPLE_TEXT if SAMPLE_SOURCE == "text" else "",
                 gt_Z=sample.image(300), gt_bounds=np.array(sample.bounds), noise_std=NOISE_STD,
                 score_bad_px=ev["bad_px"], score_edge_cov=ev["edge_cov"], score_rmse=ev["rmse"],
                 est_instrument_s=ev["est_instrument_s"])
    out_prefix = modes.save(session, SIM_OUTPUT_DIR, prefix="scan", extra=extra,
                            fallback_extent=LIVE_VIEW_EXTENT_V)
else:
    print("SAVE_RUN is False -- nothing saved")

## Adding or changing a mode

The modes live in `selfdriving_afm/modes.py`, shared with the Run notebook -- so whatever
you develop and test here is what runs on the microscope:
1. write the planner in `selfdriving_afm/planners/` -- it only talks to hardware through
   `self.executor.send(x_vec, y_vec, kind=...)` (returns z) and has a `run(start_xy)`;
2. give it a config dataclass in `config.py` and a field on `ExperimentConfig`;
3. add one `Mode(...)` entry to `modes.MODES` (planner class, config field, which live view,
   a summary function).

It then appears in `modes.list_modes()`, works as `MODALITY` in both notebooks, and can
be put in the comparison above next to the existing ones.